In [1]:
# Witness Statement of Leonardo Fabian Marañon Mejia

text = """
DRAFT TEST APPEAL SCENARIO (FOR MOLTIE PIPELINE TESTING ONLY)

Background

The Claimant was employed as a Regulatory Reporting Analyst from March 2019 until his dismissal in April 2024. He had no prior disciplinary record, had consistently received positive performance feedback, and had never been issued with a formal warning.

The Allegation

In February 2024, the Respondent alleged that the Claimant had "failed to complete a critical reconciliation task" relating to transaction reporting obligations. It was asserted that this omission exposed the firm to regulatory risk.

The Claimant denied any misconduct. He maintained that:

1. The task in question was not within his primary remit.
2. The allocation of responsibilities within the team had not been clearly defined.
3. He had repeatedly raised concerns about systemic data integrity issues which affected reconciliation outputs.

Investigation Process

The Respondent commenced an internal investigation. The investigation relied primarily on:

* A workflow log indicating that the reconciliation had not been signed off.
* An email chain suggesting that the Claimant had visibility of the issue.

However, the investigation did not:

* Examine whether the task had been formally assigned to the Claimant.
* Review system access logs to determine whether technical constraints prevented completion.
* Interview team members regarding customary allocation of reconciliation responsibilities.
* Consider whether prior systemic defects had contributed to reporting delays.

The disciplinary outcome letter concluded that the Claimant had "avoided responsibility" and demonstrated a "lack of ownership." He was summarily dismissed for gross misconduct.

Internal Appeal

On appeal, the Claimant argued that the decision had been predetermined and that the investigation had failed to consider exculpatory material. The appeal officer upheld the dismissal, stating that the original decision fell within the "range of reasonable responses" available to the employer.

Ground of Appeal

The Claimant contends that the Employment Tribunal erred in law by:

1. Substituting its own assessment of the evidence instead of applying the objective standard of a reasonable employer.
2. Failing to determine whether the Respondent had carried out as much investigation as was reasonable in the circumstances before concluding that the Claimant had avoided the task.
3. Concluding that dismissal was within the band of reasonable responses without first assessing whether the employer had reasonable grounds for its belief in misconduct.

It is submitted that a reasonable employer, faced with disputed role allocation and technical uncertainty, would have investigated the scope of duties before concluding that the Claimant had deliberately avoided a task. The failure to do so rendered the dismissal unfair under section 98(4) of the Employment Rights Act 1996.

End of test scenario.


"""

In [2]:
text = """
I was employed as a groundskeeper by the Respondent. My duties primarily involved maintaining the main estate gardens and surrounding grounds. From time to time, I was asked informally to assist with work at other locations connected to the organisation.

On one occasion, I was asked to cut and maintain grass at an additional site that was not part of my usual assignment. I carried out the work as requested. There was discussion about reimbursement for the time and equipment involved, particularly as I had used my own tools and fuel for part of the job. An invoice was later sent to cover those costs.

I did not consider this inappropriate. I understood that I had performed work at the Respondent’s request and that reimbursement for direct costs was reasonable. There was no attempt to conceal the work or the invoice.

The Respondent later treated this as serious misconduct. It was suggested that I had acted without authority and had placed the organisation in a difficult position. During the disciplinary process, further matters were raised beyond the initial concern about the invoice. I felt that the focus shifted, and that conclusions were reached without properly considering the context in which the work had been carried out.

While the process was ongoing, I made it known that I believed I was being treated unfairly. I did so because I was concerned about my professional reputation and felt the allegations were disproportionate. This was subsequently relied upon as further evidence against me.

I maintain that my actions were undertaken in good faith, that there was no dishonesty, and that summary dismissal was not a reasonable response. I further say that the process followed did not properly investigate the circumstances and fell outside the range of reasonable responses open to a reasonable employer.
"""

In [3]:
import subprocess
import textwrap

ws_text = text

proc = subprocess.run(
    [
        "python",
        "/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py",
        "--out",
        "/home/hello/Projects/Statements/output/Y_inferred.json",
        "--model",
        "mistral-small3.2:latest",
    ],
    input=ws_text,
    text=True,
    capture_output=True,
)

print(proc.stdout)
print(proc.stderr)


{
  "version": "Y_inferred_v2",
  "x_tests": {
    "X1": {
      "name": "Unauthorized Work Claim",
      "scope": "INTERNAL_DISCIPLINARY_APPEAL",
      "definition": "Claim that work was performed without proper authorization",
      "pattern": "IF the employee performed work outside usual duties without explicit authorization THEN support X1",
      "required_elements": [
        "work outside usual duties",
        "lack of explicit authorization",
        "employee performed work"
      ],
      "positive_indicators": [
        "work at additional site",
        "not part of usual assignment",
        "no authority"
      ],
      "excludes": [
        "informal request",
        "respondent's request"
      ]
    },
    "X2": {
      "name": "Reimbursement Dispute",
      "scope": "INTERNAL_DISCIPLINARY_APPEAL",
      "definition": "Dispute over reimbursement for work-related expenses",
      "pattern": "IF the employee sought reimbursement for work-related expenses and the employ

In [4]:
from pathlib import Path
import subprocess, shlex
import tqdm

NB_DIR = Path.cwd()
OUT_DIR = NB_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If your Y exists in Statements/output, point to it:
Y_SOURCE = Path("/home/hello/Projects/Statements/output/Y_inferred.json")

# Place a local copy next to the notebook (so paths stay notebook-relative)
Y_JSON = OUT_DIR / "Y_inferred.json"
if not Y_JSON.exists():
    Y_JSON.write_text(Y_SOURCE.read_text(encoding="utf-8"), encoding="utf-8")

PASS1_OUT = OUT_DIR / "pass1_results.jsonl"

BASE_DIR = Path("/home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized")
INPUT_FILES = [
    BASE_DIR / "judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_unknown_faiss_chunk_with_reasoning.jsonl",
]

import shlex
from pathlib import Path

PASS1_SCRIPT = Path("/home/hello/Projects/Statements/code/pass1_scan.py")

cmd = [
    "python", str(PASS1_SCRIPT),
    "--y-json", str(Y_JSON),
    "--out", str(PASS1_OUT),
    "--model", "mistral-small3.2:latest",
    "--ollama-url", "http://localhost:11434/api/generate",
    #"--debug-max", "200",
]

cmd += ["--input", *map(str, INPUT_FILES)]  # <-- key change

cmd_str = " ".join(shlex.quote(c) for c in cmd)
print("Running:\n", cmd_str)
!{cmd_str}



Running:
 python /home/hello/Projects/Statements/code/pass1_scan.py --y-json /home/hello/Projects/Statements/runner/output/Y_inferred.json --out /home/hello/Projects/Statements/runner/output/pass1_results.jsonl --model mistral-small3.2:latest --ollama-url http://localhost:11434/api/generate --input /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_unknown_faiss_chunk_with_reasoning.jsonl
scan judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl: 278lines [00:00, 22377.35lines/s, skipped=278, written=0]
scan judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl: 428lines [00:00, 22216.53lines/s, skipped=706, written=0]
scan judgments_unknown_faiss_chunk_with_reasoning.jsonl: 479lines [0

In [5]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "output"
PASS1_PATH = OUT_DIR / "pass1_results.jsonl"
Y_PATH = OUT_DIR / "Y_inferred.json"

# --- load pass1 ---
rows = []
with PASS1_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
df = pd.DataFrame(rows)

# --- load Y (for X-name mapping) ---
Y = json.loads(Y_PATH.read_text(encoding="utf-8"))
x_tests = (Y.get("x_tests") or {})
X_NAME = {k: (v.get("name") or "").strip() for k, v in x_tests.items()}

def matched_names(matched):
    if not isinstance(matched, list):
        return ""
    out = []
    for x in matched:
        x = str(x).strip()
        if not x:
            continue
        name = X_NAME.get(x, "")
        out.append(f"{x}: {name}" if name else x)
    return " | ".join(out)

df["matched_X_names"] = df["matched_X"].apply(matched_names) if "matched_X" in df.columns else ""

# filename guess (same logic)
def filename_guess(row):
    fn = row.get("filename")
    if isinstance(fn, str) and fn.strip():
        return fn.strip()
    item_id = row.get("item_id")
    if not isinstance(item_id, str):
        return None
    parts = item_id.split("::")
    return parts[2] if len(parts) >= 3 else None

df["filename_guess"] = df.apply(filename_guess, axis=1)

# flatten evidence snippets into a single readable field
def flatten_snips(snips, max_each=240, max_total=1200):
    if not isinstance(snips, list):
        return ""
    chunks = []
    total = 0
    for s in snips:
        if not isinstance(s, dict):
            continue
        x = str(s.get("x") or "").strip()
        txt = str(s.get("snippet") or "").strip()
        if not txt:
            continue
        txt = txt.replace("\n", " ")
        if len(txt) > max_each:
            txt = txt[:max_each] + "…"
        chunk = f"{x}: {txt}" if x else txt
        if total + len(chunk) > max_total:
            chunks.append("…")
            break
        chunks.append(chunk)
        total += len(chunk)
    return " | ".join(chunks)

df["evidence_snips_flat"] = df["evidence_snippets"].apply(flatten_snips) if "evidence_snippets" in df.columns else ""

# focus: rows with ANY match
matched_df = df[df["matched_X"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy() if "matched_X" in df.columns else df.iloc[0:0].copy()
matched_df = matched_df.sort_values("confidence", ascending=False) if "confidence" in matched_df.columns else matched_df

# columns to export/view (only keep what exists)
cols = [
    "filename_guess",
    "appeal_type",
    "who_appealed",
    "outcome",
    "successful",
    "summary_clean",
    "reasoning_for_index",
    "confidence",
    "matched_X",
    "matched_X_names",
    "evidence_snips_flat",
    "note",
    "source_file",
    "source_field",
    "item_id",
]
cols = [c for c in cols if c in matched_df.columns]

# export
matched_df.to_csv(OUT_DIR / "matched_x_results.csv", columns=cols, index=False)

# view
matched_df.head(10)[cols]


,filename_guess,appeal_type,who_appealed,outcome,successful,summary_clean,reasoning_for_index,confidence,matched_X,matched_X_names,evidence_snips_flat,note,source_file,source_field,item_id
428,Mrs_I_Shah_v_TIAA_Ltd_UKEAT_0180_19_BA.pdf,Substantive,EMPLOYEE,DISMISSED,False,The appeal concerned claims of disability disc...,The core legal issues centered on whether the ...,0,[X6],X6: Inadequate Weight to Commercial Rationale,X6: vel-related performance targets constitute...,The text mentions the dismissal was proportion...,judgments_respondent_fav_faiss_chunk_with_reas...,reasoning_for_index,judgments_respondent_fav_faiss_chunk_with_reas...
683,University_College_London_Hospitals_NHS_Founda...,Substantive,EMPLOYER,UPHELD,True,The appeal concerned proceedings brought by Mr...,The core legal issues in dispute centered on w...,0,[X6],X6: Inadequate Weight to Commercial Rationale,X6: under the Equality Act 2010 prior to Septe...,The text mentions the proportionality of dismi...,judgments_respondent_fav_faiss_chunk_with_reas...,reasoning_for_index,judgments_respondent_fav_faiss_chunk_with_reas...
1164,Vodafone_Ltd_v_Miss_A_Nicholson_UKEAT_0605_12_...,Substantive,EMPLOYER,REMITTED,False,The appeal concerned an employer's decision to...,The core legal issues centered on whether the ...,0,[X4],X4: Unfair Dismissal,X4: n whether the dism,"The text mentions 'lack of clear policies, war...",judgments_unknown_faiss_chunk_with_reasoning.j...,reasoning_for_index,judgments_unknown_faiss_chunk_with_reasoning.j...
